In [17]:
# vibary to mean a library based on vibes

In [18]:
!pip3 install sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

In [20]:
books_df = pd.read_csv('../data/BooksDatasetClean.csv')
model = SentenceTransformer('all-MiniLM-L6-v2')

# Fill missing titles or descriptions with an empty string
books_df['Title'] = books_df['Title'].fillna('')
books_df['Description'] = books_df['Description'].fillna('')
books_df['Category'] = books_df['Category'].fillna('')

text_to_encode = (books_df['Title'] + " " + books_df['Description'] + " " + books_df['Category']).to_list()
# Filter out any rows that ended up completely empty
text_to_encode = [str(text) for text in text_to_encode if text.strip() != ""]
book_embeddings = model.encode(text_to_encode, show_progress_bar=True)

Batches:   0%|          | 0/3221 [00:00<?, ?it/s]

In [21]:
np.save('books_embeddings.npy', book_embeddings)

In [22]:
from sentence_transformers.util import semantic_search

In [23]:
def vibe_search_books(query, top_k=3):
    query_embedding = model.encode(query)
    # cosine similarity
    hits = semantic_search(query_embedding, np.load('books_embeddings.npy'), top_k=top_k)
    for hit in hits[0]:
        idx = hit['corpus_id']
        print(f"Match: {books_df.iloc[idx]['Title']} (Score: {hit['score']:.2f})")

In [24]:
vibe_search_books("military")

Match: Soviet Army (Score: 0.65)
Match: The Marines (Score: 0.65)
Match: National Defense (Score: 0.64)


In [25]:
vibe_search_books("philosophy in exercising")

Match: An Integrated Approach to Therapeutic Exercise, Theory and Clinical Application (Score: 0.62)
Match: Working Out, Working Within: The Tao of Inner Fitness Through Sports and Exercise (Score: 0.61)
Match: Fitness Is Religion Keep the Faith (Score: 0.60)
